## Feature Engineering

### Understanding Feature Extraction vs. Feature Engineering

- **Feature Extraction :** Refers to extracting raw physical measurements directly from raw data sources (e.g., locating facial landmarks and measuring raw pixel distances such as `eye_distance`, `mouth_width`, `face_width`, and `face_height`).
- **Feature Engineering :** Refers to constructing new, domain-informed derived variables from the raw measurements. Raw pixel distances vary depending on camera zoom, cropping, and resolution. Feature engineering addresses this scale dependency by deriving **proportional ratios**, which are **scale-invariant** and reflect underlying facial structure regardless of image resolution.

In [1]:
# Ensure working directory is set to project root
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Imports for data processing and numerical computations
import pandas as pd
import numpy as np

# Load dataset with raw facial measurements
measurements_csv_path = "data/processed/utkface_with_measurements.csv"
df = pd.read_csv(measurements_csv_path)

print("=== BEFORE DROPPING FAILED EXTRACTIONS ===")
print(f"Loaded dataset shape: {df.shape}")
measurement_cols = ['eye_distance', 'mouth_width', 'face_width', 'face_height']
print("\nNull counts for measurement columns:")
print(df[measurement_cols].isnull().sum())

=== BEFORE DROPPING FAILED EXTRACTIONS ===
Loaded dataset shape: (3000, 14)

Null counts for measurement columns:
eye_distance    2
mouth_width     2
face_width      2
face_height     2
dtype: int64


In [2]:
# Drop rows where any measurement column contains NaN (failed landmark extractions)
initial_row_count = len(df)
df_clean = df.dropna(subset=measurement_cols).reset_index(drop=True)
final_row_count = len(df_clean)
rows_dropped = initial_row_count - final_row_count

print("=== AFTER DROPPING FAILED EXTRACTIONS ===")
print(f"Cleaned dataset shape: {df_clean.shape}")
print(f"Total rows dropped due to failed landmark detection: {rows_dropped}")

=== AFTER DROPPING FAILED EXTRACTIONS ===
Cleaned dataset shape: (2998, 14)
Total rows dropped due to failed landmark detection: 2


### Engineered Ratio Features Design

To ensure measurements are scale-invariant across varying image crops and resolutions, three key ratio features are derived:

1. **`face_aspect_ratio`**:
   - **Original Features Used:** `face_height`, `face_width`
   - **Formula:** `face_aspect_ratio = face_height / face_width`
   - **Engineering Reason:** Captures overall facial morphology (elongated vs. round face shape) independently of absolute image dimensions or bounding box scale.

2. **`eye_to_face_ratio`**:
   - **Original Features Used:** `eye_distance`, `face_width`
   - **Formula:** `eye_to_face_ratio = eye_distance / face_width`
   - **Engineering Reason:** Normalizes inter-ocular spacing relative to overall facial width, allowing direct comparison across subject faces regardless of distance from camera.

3. **`mouth_to_face_ratio`**:
   - **Original Features Used:** `mouth_width`, `face_width`
   - **Formula:** `mouth_to_face_ratio = mouth_width / face_width`
   - **Engineering Reason:** Normalizes mouth width relative to overall facial width to quantify facial component proportions in a scale-invariant manner.

In [3]:
# Compute scale-invariant ratio features using vectorized Pandas operations
df_clean['face_aspect_ratio'] = df_clean['face_height'] / df_clean['face_width']
df_clean['eye_to_face_ratio'] = df_clean['eye_distance'] / df_clean['face_width']
df_clean['mouth_to_face_ratio'] = df_clean['mouth_width'] / df_clean['face_width']

In [4]:
print("=== ENGINEERED FEATURE STATISTICS ===")
ratio_cols = ['face_aspect_ratio', 'eye_to_face_ratio', 'mouth_to_face_ratio']
print(df_clean[ratio_cols].describe())

print("\nDataFrame Head with Engineered Features:")
df_clean.head()

=== ENGINEERED FEATURE STATISTICS ===
       face_aspect_ratio  eye_to_face_ratio  mouth_to_face_ratio
count        2998.000000        2998.000000          2998.000000
mean            1.156118           0.465162             0.388944
std             0.078710           0.024219             0.036520
min             0.924732           0.316026             0.211136
25%             1.104011           0.448179             0.364624
50%             1.161063           0.463607             0.391353
75%             1.208588           0.481006             0.415991
max             1.469383           0.545958             0.484194


,image_name,age,gender,race,filepath,gender_0,gender_1,race_0,race_1,race_2,race_3,race_4,eye_distance,mouth_width,face_width,face_height,face_aspect_ratio,eye_to_face_ratio,mouth_to_face_ratio
0,21_0_2_20170116170741864.jpg.chip.jpg,21,0,2,/Users/cipherxishant/Downloads/archive/UTKFace...,1,0,0,0,1,0,0,77.408775,66.862417,160.524368,193.195435,1.203527,0.482225,0.416526
1,39_0_4_20170104205430619.jpg.chip.jpg,39,0,4,/Users/cipherxishant/Downloads/archive/UTKFace...,1,0,0,0,0,0,1,81.252033,67.094595,172.952957,196.229156,1.134581,0.469793,0.387948
2,36_0_4_20170104000906228.jpg.chip.jpg,36,0,4,/Users/cipherxishant/Downloads/archive/UTKFace...,1,0,0,0,0,0,1,81.890691,70.198944,174.282593,198.857635,1.141007,0.469873,0.402788
3,26_0_3_20170119150512262.jpg.chip.jpg,26,0,3,/Users/cipherxishant/Downloads/archive/UTKFace...,1,0,0,0,0,1,0,80.893077,64.496580,176.753891,194.043320,1.097816,0.457659,0.364895
4,21_1_1_20170112192949478.jpg.chip.jpg,21,1,1,/Users/cipherxishant/Downloads/archive/UTKFace...,0,1,0,1,0,0,0,80.203875,73.080556,169.878937,192.511398,1.133227,0.472124,0.430191


In [5]:
# Export engineered DataFrame to data/processed/utkface_engineered.csv
output_engineered_path = "data/processed/utkface_engineered.csv"
df_clean.to_csv(output_engineered_path, index=False)

print(f"Engineered feature dataset saved successfully to '{output_engineered_path}' ({len(df_clean)} rows, {len(df_clean.columns)} columns).")

Engineered feature dataset saved successfully to 'data/processed/utkface_engineered.csv' (2998 rows, 19 columns).
